**Dependencies**

In [ ]:

!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate peft datasets bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00


**Imports and setup**

In [ ]:
import os
import random
import numpy as np
import torch
import pandas as pd

from transformers import CLIPTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


**Reproducability**

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

**Load captions.csv**

In [ ]:
df = pd.read_csv("captions.csv")
print("Total samples:", len(df))
df.head()

Total samples: 5000


,image,caption
0,/content/dataset/leftImg8bit/train/krefeld/kre...,The image shows a car driving down a street li...
1,/content/dataset/leftImg8bit/train/krefeld/kre...,The image shows a street with cars parked on t...
2,/content/dataset/leftImg8bit/train/krefeld/kre...,The image shows a car driving down a street li...
3,/content/dataset/leftImg8bit/train/krefeld/kre...,The image shows a man riding a bicycle down a ...
4,/content/dataset/leftImg8bit/train/krefeld/kre...,The image shows a person riding a bicycle down...


**Filter captions over 77 tokens**

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

def token_length(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

df["token_length"] = df["caption"].apply(token_length)

filtered_df = df[df["token_length"] <= 77].copy()

print("After filtering:", len(filtered_df))
print("Removed:", len(df) - len(filtered_df))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (78 > 77). Running this sequence through the model will result in indexing errors


After filtering: 4995
Removed: 5


**Download the dataset**

In [ ]:
import zipfile
import os

# Install gdown (if not already)
!pip install -q gdown

# Download dataset
!gdown "https://drive.google.com/uc?id=1qYNQgfEFuPa7L75TeuaO_Exc_rFBDF68" -O dataset.zip

zip_path = "/content/dataset.zip"
extract_path = "/content/dataset"

# Extract dataset
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

**Create dataset directory**

In [ ]:
import shutil
import json
import os

os.makedirs("dataset", exist_ok=True)

metadata = []

for _, row in filtered_df.iterrows():
    src = row["image"]
    filename = os.path.basename(src)
    dst = os.path.join("dataset", filename)  # 🔥 NO /images/

    if os.path.exists(src):
        shutil.copy(src, dst)

        metadata.append({
            "file_name": filename,
            "text": row["caption"]
        })

# Save metadata.jsonl
with open("dataset/metadata.jsonl", "w") as f:
    for entry in metadata:
        f.write(json.dumps(entry) + "\n")

print("Dataset prepared!")

In [ ]:
print("Images:", len(os.listdir("dataset/images")))

with open("dataset/metadata.jsonl") as f:
    for i in range(3):
        print(next(f))

**Lora adaptation script**

In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py

In [ ]:
!accelerate config default

**Train**

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="./dataset" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=2000 \
  --checkpointing_steps=500 \
  --mixed_precision="fp16" \
  --output_dir="lora-output"